# LPP Differential Expression Analysis

This notebook performs differential expression analysis of untreated vs LPP-treated samples using Teton cytoprofiling output. 

It performs:
- cell filtering and normalization across all selected case+control samples together
- heatmap plotting for inter-well R2 correlation
- per-target unpaired t-tests on well-level log2 values
- BH (Benjamini-Hochberg) multiple-testing correction
- scatter and volcano plots with significant target labeling (top 10 labels max)

In order to use the notebook, first download and unpack the public dataset located at https://element-public-data.s3.us-west-2.amazonaws.com/cytoprofiling/ACE-3376.tar. Update the "Inputs and thresholds" cell below to point to the location of the downloaded data. 

Ensure the following packages listed below are available in your Python installation
* scipy
* statsmodels
* matplotlib
* numpy
* pandas
* cytoprofiling

The cytoprofiling package is provided by Element Biosciences specifically for the analysis of Teton cytoprofiling data. Installation instructions for the latest version of this package are available at https://gitlab.com/elembio/analysis/cytoprofiling/-/blob/main/README.md?ref_type=heads#installation. 


After all data is downloaded and dependencies are installed, run the sections in order from top to bottom.

## Inputs and thresholds

Set file paths, run metadata, labels to compare, and significance cutoffs used for labeling targets in plots.

Update these values before running downstream cells.

In [ ]:
# Use raw strings, escaped backslashes, or forward slashes for Windows paths.
cell_stats_file = "ACE-3376/Cytoprofiling/Instrument/RawCellStats.parquet"
run_name = "ACE-3376"

# RNA batches to include in filtering/normalization
batch_names = ["B01"]

# Group labels in WellLabel column (single label each, may map to multiple wells)
control_label = "WT"
case_label = "LPP"

# Significance settings
fdr_threshold = 0.05
abs_log2fc_threshold = 0.25
max_labeled_targets = 5

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from matplotlib import pyplot as plt
from cytoprofiling import (
    get_wells,
    filter_cells,
    get_default_normalization_targets,
    normalize_cells_by_aggregated_counts,
    normalize_wells_by_median_of_ratios,
)

## Load data, filter cells, and select case/control wells

This section loads cell-level data, applies default filtering, and identifies all wells that belong to the selected `control_label` and `case_label`.

It also validates that each group has wells available.

In [ ]:
os.makedirs(run_name, exist_ok=True)

df = pd.read_parquet(cell_stats_file)
df = filter_cells(df, batch_names=batch_names, stats={})

well2label = {}
for well in get_wells(df):
    labels = df.loc[df["Well"] == well, "WellLabel"].dropna().unique()
    if len(labels) > 0:
        well2label[well] = labels[0]

control_wells = [well for well, label in well2label.items() if label == control_label]
case_wells = [well for well, label in well2label.items() if label == case_label]
target_wells = control_wells + case_wells

if len(control_wells) == 0:
    raise ValueError(f"No wells found for control_label='{control_label}' after filtering.")
if len(case_wells) == 0:
    raise ValueError(f"No wells found for case_label='{case_label}' after filtering.")

if len(control_wells) < 2 or len(case_wells) < 2:
    warnings.warn(
        "Fewer than 2 wells in one or both groups. "
        "Unpaired t-tests may be undefined; p-values may be NaN."
    )

target_df = df.loc[df["Well"].isin(target_wells)].copy().reset_index(drop=True)

print(f"Control wells ({control_label}): {len(control_wells)}")
print(f"Case wells ({case_label}): {len(case_wells)}")
print(f"Total filtered cells in analysis: {len(target_df)}")

## Normalize all selected samples together

This section applies the same normalization workflow to all case and control wells in a single combined dataset.

Targets are collected after normalization for downstream testing.

In [ ]:
norm_df = normalize_cells_by_aggregated_counts(target_df, batch_names=batch_names)
well_names = get_wells(norm_df)
norm_df = normalize_wells_by_median_of_ratios(
    norm_df,
    batch_names=batch_names,
    well_names=well_names,
)

all_targets = []
for batch_name in batch_names:
    all_targets.extend(get_default_normalization_targets(norm_df, batch_name))

# Keep unique target order
all_targets = list(dict.fromkeys(all_targets))
print(f"Number of targets tested: {len(all_targets)}")

## Sample-to-sample R<sup>2</sup> heatmap

This section summarizes sample similarity after normalization.

- Samples are compared at the well level
- Feature values are averaged per well, then log2-transformed
- Pairwise Pearson correlations are computed between wells and squared (R<sup>2</sup>)
- Wells are ordered by `WellLabel` so same-label samples are grouped together

In [ ]:
# Build well-level matrix (samples = wells, features = targets), then log-transform.
well_target_means = norm_df.groupby("Well")[all_targets].mean()
well_target_log2 = np.log2(well_target_means)

# Map each well to its WellLabel and order wells by label, then well name.
well_label_map = (
    norm_df[["Well", "WellLabel"]]
    .dropna(subset=["Well", "WellLabel"])
    .drop_duplicates(subset=["Well"])
    .set_index("Well")["WellLabel"]
)

well_order_df = pd.DataFrame({
    "Well": well_target_log2.index,
    "WellLabel": well_target_log2.index.map(well_label_map),
})
well_order_df["WellLabel"] = well_order_df["WellLabel"].fillna("Unknown")
well_order_df = well_order_df.sort_values(["WellLabel", "Well"]).reset_index(drop=True)
ordered_wells = well_order_df["Well"].tolist()

well_target_log2 = well_target_log2.loc[ordered_wells]

# Correlation is computed on log-transformed values across targets.
r_matrix = well_target_log2.T.corr(method="pearson")
r2_matrix = r_matrix ** 2

# Plot heatmap.
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(r2_matrix.to_numpy(dtype=float), cmap="viridis", vmin=0.75, vmax=1)

# Tick labels include both label and well for readability.
tick_labels = [f"{lbl}|{well}" for lbl, well in zip(well_order_df["WellLabel"], well_order_df["Well"])]
ax.set_xticks(np.arange(len(tick_labels)))
ax.set_yticks(np.arange(len(tick_labels)))
ax.set_xticklabels(tick_labels, rotation=90, fontsize=8)
ax.set_yticklabels(tick_labels, fontsize=8)
ax.set_title(f"{run_name}: Sample-to-sample R2")

# Draw group boundaries where WellLabel changes.
label_change_idx = np.where(well_order_df["WellLabel"].to_numpy()[:-1] != well_order_df["WellLabel"].to_numpy()[1:])[0]
for idx in label_change_idx:
    boundary = idx + 0.5
    ax.axhline(boundary, color="white", linewidth=1)
    ax.axvline(boundary, color="white", linewidth=1)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("R2")
fig.tight_layout()

r2_path = f"{run_name}/sample_to_sample_r2_heatmap.png"
fig.savefig(r2_path, dpi=150)
plt.show()
plt.close(fig)

print(f"R2 heatmap: {r2_path}")

## Compute differential expression statistics

For each target:
- compute log2 of well-level mean values
- compute average control and case signal
- run an unpaired t-test on well-level log2 values
- adjust p-values with Benjamini-Hochberg (BH)
- flag significant targets using notebook thresholds

Results are exported to a CSV file for review.

In [ ]:
records = []
for target in all_targets:
    per_well_control = []
    for well in control_wells:
        vals = norm_df.loc[norm_df["Well"] == well, target].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            mean_val = float(np.nanmean(vals))
            per_well_control.append(np.log2(mean_val))

    per_well_case = []
    for well in case_wells:
        vals = norm_df.loc[norm_df["Well"] == well, target].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) > 0:
            mean_val = float(np.nanmean(vals))
            per_well_case.append(np.log2(mean_val))

    control_mean = np.nanmean(per_well_control) if len(per_well_control) > 0 else np.nan
    case_mean = np.nanmean(per_well_case) if len(per_well_case) > 0 else np.nan
    log2_fc = case_mean - control_mean

    if len(per_well_control) >= 2 and len(per_well_case) >= 2:
        test_result = ttest_ind(per_well_case, per_well_control, equal_var=False, nan_policy="omit")
        p_value = float(test_result.pvalue)
    else:
        p_value = np.nan

    records.append({
        "Target": target,
        "MeanLog2Control": control_mean,
        "MeanLog2Case": case_mean,
        "Log2FC_CaseMinusControl": log2_fc,
        "PValue": p_value,
        "NControlWellsUsed": len(per_well_control),
        "NCaseWellsUsed": len(per_well_case),
    })

results_df = pd.DataFrame(records)
valid_mask = np.isfinite(results_df["PValue"].to_numpy(dtype=float))
adj_pvals = np.full(len(results_df), np.nan)
if valid_mask.any():
    adj_pvals[valid_mask] = multipletests(
        results_df.loc[valid_mask, "PValue"].to_numpy(dtype=float),
        alpha=fdr_threshold,
        method="fdr_bh",
    )[1]
results_df["AdjPValue_BH"] = adj_pvals
results_df["Significant"] = (
    (results_df["AdjPValue_BH"] <= fdr_threshold)
    & (np.abs(results_df["Log2FC_CaseMinusControl"]) >= abs_log2fc_threshold)
)

results_df = results_df.sort_values(["AdjPValue_BH", "PValue"], na_position="last").reset_index(drop=True)
results_path = f"{run_name}/group_differential_expression.csv"
results_df.to_csv(results_path, index=False)

sig_df = results_df.loc[results_df["Significant"]].copy()
label_df = sig_df.nsmallest(max_labeled_targets, "AdjPValue_BH")

print(f"Significant targets: {len(sig_df)}")
print(f"Labeled targets on plots: {len(label_df)}")
print(f"Results table: {results_path}")

## Generate scatter and volcano plots

- Scatter plot: average control vs average case log2 values
- Volcano plot: log2 fold-change vs `-log10(BH-adjusted p-value)`

Significant targets are highlighted. Up to `max_labeled_targets` top differentially expressed targets (by adjusted p-value among significant targets) are labeled. Targets beginning with `phos` are plotted with a different marker so they are easy to inspect for potential false negatives.

Plot files are saved under the run output directory.

In the plots of differential expression below, the expectation is that the levels of phosphorylated proteins should be reduced by the LPP treatment. In practice, the majority of phosphorylated targets (triangles) show a reduction in counts as expected. This reduction in counts is also specific, with only phosphorylated protein targets showing absolute log2 fold change greater than 0.25. Of the nine phosphorylated targets not showing log fold change greater than 0.25, six (phosS100-JunD_phosS73-cJun, phosS10-p27, phosS439-TAK1, phosS46-p53, phosS393-PDPK1, phosT210-Plk1) are expected to show limited reduction in counts following LPP treatment due to limited presence of the phosphorylated version of the protein in the untreated condition. 

In [ ]:
plot_df = results_df.copy()
plot_df["NegLog10AdjP"] = -np.log10(np.clip(plot_df["AdjPValue_BH"].to_numpy(dtype=float), 1e-300, 1.0))

phos_mask = plot_df["Target"].str.startswith("phos", na=False)
label_plot_df = (
    plot_df.loc[plot_df["Significant"]]
    .nsmallest(max_labeled_targets, "AdjPValue_BH")
    .copy()
)

non_phos_non_sig = plot_df.loc[~plot_df["Significant"] & ~phos_mask]
phos_non_sig = plot_df.loc[~plot_df["Significant"] & phos_mask]
non_phos_sig = plot_df.loc[plot_df["Significant"] & ~phos_mask]
phos_sig = plot_df.loc[plot_df["Significant"] & phos_mask]

# Scatter plot
plt.figure(figsize=(8, 8))
if len(non_phos_non_sig) > 0:
    plt.scatter(
        non_phos_non_sig["MeanLog2Control"],
        non_phos_non_sig["MeanLog2Case"],
        s=20,
        alpha=0.6,
        marker="o",
        label="Not significant",
    )
if len(non_phos_sig) > 0:
    plt.scatter(
        non_phos_sig["MeanLog2Control"],
        non_phos_sig["MeanLog2Case"],
        s=28,
        alpha=0.9,
        marker="o",
        label="Significant",
    )
if len(phos_non_sig) > 0:
    plt.scatter(
        phos_non_sig["MeanLog2Control"],
        phos_non_sig["MeanLog2Case"],
        s=38,
        alpha=0.9,
        marker="^",
        facecolors="none",
        edgecolors="tab:blue",
        label="phos (not significant)",
    )
if len(phos_sig) > 0:
    plt.scatter(
        phos_sig["MeanLog2Control"],
        phos_sig["MeanLog2Case"],
        s=45,
        alpha=0.9,
        marker="^",
        c="tab:orange",
        label="phos (significant)",
    )

min_val = np.nanmin([plot_df["MeanLog2Control"].min(), plot_df["MeanLog2Case"].min()])
max_val = np.nanmax([plot_df["MeanLog2Control"].max(), plot_df["MeanLog2Case"].max()])
plt.plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--", linewidth=1)

scatter_texts = []
for _, row in label_plot_df.iterrows():
    scatter_texts.append(
        plt.text(
            row["MeanLog2Control"],
            row["MeanLog2Case"],
            row["Target"],
            fontsize=8,
        )
    )

plt.xlabel(f"Average log2 {control_label}")
plt.ylabel(f"Average log2 {case_label}")
plt.title(f"{run_name}: average control vs case")
plt.legend()
plt.tight_layout()
scatter_path = f"{run_name}/scatter_control_vs_case.png"
plt.savefig(scatter_path, dpi=150)
plt.show()
plt.close()

# Volcano plot
plt.figure(figsize=(9, 7))
if len(non_phos_non_sig) > 0:
    plt.scatter(
        non_phos_non_sig["Log2FC_CaseMinusControl"],
        non_phos_non_sig["NegLog10AdjP"],
        s=20,
        alpha=0.6,
        marker="o",
        label="Not significant",
    )
if len(non_phos_sig) > 0:
    plt.scatter(
        non_phos_sig["Log2FC_CaseMinusControl"],
        non_phos_sig["NegLog10AdjP"],
        s=28,
        alpha=0.9,
        marker="o",
        label="Significant",
    )
if len(phos_non_sig) > 0:
    plt.scatter(
        phos_non_sig["Log2FC_CaseMinusControl"],
        phos_non_sig["NegLog10AdjP"],
        s=38,
        alpha=0.9,
        marker="^",
        facecolors="none",
        edgecolors="tab:blue",
        label="phos (not significant)",
    )
if len(phos_sig) > 0:
    plt.scatter(
        phos_sig["Log2FC_CaseMinusControl"],
        phos_sig["NegLog10AdjP"],
        s=45,
        alpha=0.9,
        marker="^",
        c="tab:orange",
        label="phos (significant)",
    )

plt.axvline(abs_log2fc_threshold, color="black", linestyle="--", linewidth=1)
plt.axvline(-abs_log2fc_threshold, color="black", linestyle="--", linewidth=1)
plt.axhline(-np.log10(fdr_threshold), color="black", linestyle="--", linewidth=1)

volcano_texts = []
for _, row in label_plot_df.iterrows():
    volcano_texts.append(
        plt.text(
            row["Log2FC_CaseMinusControl"],
            row["NegLog10AdjP"],
            row["Target"],
            fontsize=8,
        )
    )

plt.xlabel("Log2 fold change (case - control)")
plt.ylabel("-log10(p-value)")
plt.title(f"{run_name}: volcano plot")
plt.legend()
plt.tight_layout()
volcano_path = f"{run_name}/volcano_case_vs_control.png"
plt.savefig(volcano_path, dpi=150)
plt.show()
plt.close()

print(f"Scatter plot: {scatter_path}")
print(f"Volcano plot: {volcano_path}")